In [1]:
import os

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["WANDB__SERVICE_WAIT"] = "300"

import wandb

In [1]:
# Cell 0. Install dependencies
!pip install -q wandb pycocotools

In [2]:
# Cell 1. Google Drive mount
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Cell 1. Project setup
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/content/project01")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("cwd:", os.getcwd())
print("files:", os.listdir(PROJECT_ROOT))

cwd: /content/project01
files: ['checkpoints', 'submit', 'src', 'main.ipynb']


In [4]:
print(os.listdir("/content"))
print(os.listdir("/content/project01"))

['.config', 'drive', 'project01', 'sample_data']
['checkpoints', 'submit', 'src', 'main.ipynb']


In [5]:
import shutil
from pathlib import Path

Path("/content/project01").mkdir(exist_ok=True)

for name in ["src", "checkpoints", "submit", "reports", "pyproject.toml", "README.md"]:
    src = Path("/content") / name
    dst = Path("/content/project01") / name
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))

In [6]:
# shutil.move('/content/default.yaml', '/content/project01/src/config/default.yaml')

In [7]:
# Cell 2. Install
!pip install -q pycocotools wandb thop

In [7]:
# Cell 3. Config + WandB
from src.config.config import load_config
from src.utils.logger import init_wandb, print_config

cfg = load_config("src/config/default.yaml")
print_config(cfg)

wandb_run = init_wandb(cfg)

Config
model: {'name': 'fpn_mobilenet', 'backbone': 'mobilenet_v3_large', 'dropout': 0.1, 'num_classes': 21, 'pretrained': True, 'fpn_channels': 128}
data: {'dataset_name': 'VOC', 'num_workers': 4, 'pin_memory': True, 'input_size': 512, 'num_classes': 21, 'use_coco_mask_cache': True, 'coco_mask_cache_dir': '/content/datasets/coco/coco_voc_mask_cache', 'root': '/content/datasets', 'coco_root': '/content/datasets/coco/train2017', 'coco_ann_file': '/content/datasets/coco/annotations/instances_train2017.json', 'train_datasets': [{'name': 'VOC', 'year': '2007', 'split': 'train'}, {'name': 'VOC', 'year': '2012', 'split': 'train'}, {'name': 'coco_voc', 'split': 'train'}], 'val_dataset': {'name': 'VOC', 'year': '2012', 'split': 'val'}}
training: {'batch_size': 16, 'epochs': 60, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'lr_decay': 0.9995, 'ignore_index': 255, 'eval_interval': 2}
checkpoint: {'save_dir': '/content/drive/MyDrive/project1_checkpoints', 'save_every': 5, 'resume_path': '/cont

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jyun_chae (jyun_chae-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
from pathlib import Path
import shutil

COCO_ROOT = Path("/content/datasets/coco")
ANN_ZIP = COCO_ROOT / "annotations_trainval2017.zip"
ANN_DIR = COCO_ROOT / "annotations"

if ANN_ZIP.exists():
    ANN_ZIP.unlink()
    print("removed broken zip:", ANN_ZIP)

# annotations 폴더가 깨진 상태일 수도 있으니 필요하면 삭제
if ANN_DIR.exists() and not (ANN_DIR / "instances_train2017.json").exists():
    shutil.rmtree(ANN_DIR)
    print("removed incomplete annotations dir:", ANN_DIR)

!wget -O /content/datasets/coco/annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -o /content/datasets/coco/annotations_trainval2017.zip -d /content/datasets/coco

print("COCO annotation:", (COCO_ROOT / "annotations/instances_train2017.json").exists())
print("COCO images:", (COCO_ROOT / "train2017").exists(), len(list((COCO_ROOT / "train2017").glob("*.jpg"))))

removed broken zip: /content/datasets/coco/annotations_trainval2017.zip
--2026-04-26 09:13:43--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.223.149, 52.216.49.145, 16.15.191.81, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.223.149|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/content/datasets/coco/annotations_trainval2017.zip’

/content/datasets/c 100%[===================>] 241.19M  15.1MB/s    in 18s     

2026-04-26 09:14:01 (13.5 MB/s) - ‘/content/datasets/coco/annotations_trainval2017.zip’ saved [252907541/252907541]

COCO annotation: True
COCO images: True 118287


In [9]:
# Cell. Prepare COCO 2017
from pathlib import Path

COCO_ROOT = Path("/content/datasets/coco")
ANN_PATH = COCO_ROOT / "annotations" / "instances_train2017.json"
IMG_DIR = COCO_ROOT / "train2017"

COCO_ROOT.mkdir(parents=True, exist_ok=True)

if not ANN_PATH.exists():
    !wget -nc -P /content/datasets/coco http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -n /content/datasets/coco/annotations_trainval2017.zip -d /content/datasets/coco

if not IMG_DIR.exists() or len(list(IMG_DIR.glob("*.jpg"))) == 0:
    !wget -nc -P /content/datasets/coco http://images.cocodataset.org/zips/train2017.zip
    !unzip -q -n /content/datasets/coco/train2017.zip -d /content/datasets/coco

print("COCO annotation:", ANN_PATH.exists())
print("COCO images:", IMG_DIR.exists(), len(list(IMG_DIR.glob('*.jpg'))) if IMG_DIR.exists() else 0)

--2026-04-26 08:42:52--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.217.143.49, 54.231.196.185, 16.182.65.249, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.217.143.49|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/content/datasets/coco/annotations_trainval2017.zip’

   annotations_trai  75%[==============>     ] 183.02M  17.3MB/s    eta 4s     ^C
[/content/datasets/coco/annotations_trainval2017.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/datasets/coco/annotations_trainval2017.zip or
        /content/datasets/coco/annotations_trainv

In [10]:
# Cell 4. Runtime
import torch

device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
print("device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device: cuda
NVIDIA L4


In [11]:
!grep -n "GradScaler\|scaler" /content/project01/src/train.py

23:    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
82:            scaler=scaler


In [14]:
# Cell 5. Train
from src.train import main as train_main

train_main(cfg)

[2026-04-26 09:14:16] [INFO] Using device: cuda


loading annotations into memory...
Done (t=12.03s)
creating index...
index created!
[COCOVOCSegDataset] total images: 118287, valid images: 95279, use_cache: True, cache_dir: /content/datasets/coco/coco_voc_mask_cache
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 226MB/s]
[2026-04-26 09:15:02] [INFO] Epoch [16/60]
Train Epoch 15:   0%|          | 0/6059 [00:00<?, ?it/s]/content/project01/src/engine/trainer.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[2026-04-26 09:29:48] [INFO] Train Loss: 0.4035                                   
[2026-04-26 09:30:00] [INFO] Val Loss: 0.5182                                    
[2026-04-26 09:30:00] [INFO] Val mIoU: 0.3560
[2026-04-26 09:30:00] [INFO] Val mIoU: 0.3560
[2026-04-26 09:30:00] [INFO] Epoch [17/60]
[2026-04-26 09:42:43] [INFO] Train Loss: 0.4026                                 
[2026-04-26 09:42:44] [INFO] Epoch [18/60]
[2026-04-26 09:55:27] [INFO] Train Loss: 0.4010                                 
[2026-04-26 09:55:34] [INFO] Val Loss: 0.4881                                    
[2026-04-26 09:55:34] [INFO] Val mIoU: 0.3952
[2026-04-26 09

: 

In [ ]:
# Cell 6. Evaluate best checkpoint
from src.eval import main as eval_main

eval_main(cfg)

In [ ]:
# Cell 7. FLOPs
from src.flops import main as flops_main

flops_main(cfg)

In [ ]:
# Cell 8. Inference for submission
from src.infer import main as infer_main

infer_main(cfg)